# Train Commutative CNN Classifier

This notebook trains the **pure-CNN dual-pathway commutative classifier** on the zebrafish tensor dataset. To avoid leakage, it loads the persisted base dataset from notebook 5, splits train/validation/holdout on `original_instance_id`, and only then applies random XY rotation augmentation to the training subset.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report

from zebrafish_ml import (
    ZebrafishCommutativeCNNClassifier,
    augment_training_tensors_with_rotations,
    plot_confusion_matrices,
    plot_training_history,
    split_labeled_tensor_dataset_by_instance,
)
from zebrafish_notebook_utils import configure_full_dataframe_display
from zebrafish_tensor_utils import (
    build_tensor_embedding_2d,
    load_labeled_tensor_dataset,
    plot_tensor_embedding_2d,
)

warnings.filterwarnings("ignore", message="IProgress not found.*")


In [ ]:
configure_full_dataframe_display()

In [ ]:
# User inputs
# dataset_artifact_path:
#   relative path under .dataset_cache or an absolute path for the persisted dataset artifact from notebook 5.
#   Dataset composition is read from the saved metadata instead of being repeated in this notebook.
dataset_artifact_path = "moa_4class_t10_z3_y128_x128.pt"
# random_seed:
#   seed used for reproducible split assignment and training-time augmentation angles.
random_seed = 0

# Split and augmentation inputs
holdout_fraction = 0.25
validation_fraction_within_train = 0.2
train_num_random_rotations = 2
rotation_range_degrees = 5.0

# Commutative CNN inputs
spatial_conv_channels = (16, 32, 64)
spatial_kernel_size_z = (1, 1, 1)
spatial_kernel_size_xy = (5, 3, 3)
spatial_stride_z = (1, 1, 1)
spatial_stride_xy = (1, 1, 1)
spatial_pool_kernel_z = (1, 1, 1)
spatial_pool_kernel_xy = (2, 2, 2)
spatial_pool_stride_z = (1, 1, 1)
spatial_pool_stride_xy = (2, 2, 2)

temporal_st_channels = (128, 128)
temporal_st_kernel_sizes = (3, 3)
temporal_ts_channels = (64, 64)
temporal_ts_kernel_sizes = (5, 5)

spatial_agg_channels = (64, 128)
spatial_agg_kernel_size_z = (3, 3)
spatial_agg_kernel_size_xy = (3, 3)
spatial_agg_stride_z = (1, 1)
spatial_agg_stride_xy = (1, 1)
spatial_agg_pool_kernel_z = (1, 1)
spatial_agg_pool_kernel_xy = (2, 2)
spatial_agg_pool_stride_z = (1, 1)
spatial_agg_pool_stride_xy = (2, 2)

patch_size_z = 1
patch_size_xy = 16
embedding_dim = 128
num_prototypes = 64
consistency_weight = 0.5
feature_weight = 0.1
prototype_temperature = 0.1
dropout = 0.2

# Optimizer / training inputs
batch_size = 16
epochs = 20
learning_rate = 1e-3
weight_decay = 1e-4
device = None

# Embedding plot inputs
embedding_method = "umap"
umap_n_neighbors = 32
umap_min_dist = 0.1


In [ ]:
# Load the base dataset persisted by notebook 5 so we split first and avoid rebuilding tensors here.
dataset = load_labeled_tensor_dataset(dataset_artifact_path)
metadata_all = dataset["metadata"].reset_index(drop=True).copy()

selected_mechanisms = [
    name for label, name in sorted(dataset["label_map"].items()) if int(label) != 0
]
selected_concentrations = sorted(
    metadata_all.loc[~metadata_all["is_control"], "concentration_band"].dropna().unique().tolist()
)
dataset_summary_df = pd.DataFrame(
    [
        {"metric": "dataset_artifact_path", "value": str(dataset_artifact_path)},
        {"metric": "n_examples", "value": int(dataset["tensors"].shape[0])},
        {"metric": "tensor_shape", "value": tuple(dataset["tensors"].shape)},
        {"metric": "n_classes_including_water", "value": int(len(dataset["label_map"]))},
        {"metric": "selected_mechanisms", "value": selected_mechanisms},
        {"metric": "selected_concentrations", "value": selected_concentrations},
        {"metric": "n_original_instances", "value": int(metadata_all["original_instance_id"].nunique()) if "original_instance_id" in metadata_all.columns else None},
    ]
)

display(dataset_summary_df)
display(metadata_all.groupby(["label", "label_name"]).size().reset_index(name="n_examples"))

In [ ]:
splits = split_labeled_tensor_dataset_by_instance(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    random_state=random_seed,
)
metadata_all = splits.metadata_all
train_instance_ids = splits.train_instance_ids
val_instance_ids = splits.val_instance_ids
holdout_instance_ids = splits.holdout_instance_ids

X_train_base = splits.X_train_base
y_train_base = splits.y_train_base
metadata_train_base = splits.metadata_train_base

X_val = splits.X_val
y_val = splits.y_val
metadata_val = splits.metadata_val

X_holdout = splits.X_holdout
y_holdout = splits.y_holdout
metadata_holdout = splits.metadata_holdout

X_train, y_train, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    y_train_base,
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=random_seed,
)

split_summary_df = pd.DataFrame(
    [
        {"split": "train_base", "n_examples": int(len(X_train_base)), "n_original_instances": int(len(train_instance_ids))},
        {"split": "train_augmented", "n_examples": int(len(X_train)), "n_original_instances": int(len(train_instance_ids))},
        {"split": "val", "n_examples": int(len(X_val)), "n_original_instances": int(len(val_instance_ids))},
        {"split": "holdout", "n_examples": int(len(X_holdout)), "n_original_instances": int(len(holdout_instance_ids))},
    ]
)
display(split_summary_df)

In [ ]:
model = ZebrafishCommutativeCNNClassifier(
    spatial_conv_channels=spatial_conv_channels,
    spatial_kernel_size_z=spatial_kernel_size_z,
    spatial_kernel_size_xy=spatial_kernel_size_xy,
    spatial_stride_z=spatial_stride_z,
    spatial_stride_xy=spatial_stride_xy,
    spatial_pool_kernel_z=spatial_pool_kernel_z,
    spatial_pool_kernel_xy=spatial_pool_kernel_xy,
    spatial_pool_stride_z=spatial_pool_stride_z,
    spatial_pool_stride_xy=spatial_pool_stride_xy,
    temporal_st_channels=temporal_st_channels,
    temporal_st_kernel_sizes=temporal_st_kernel_sizes,
    temporal_ts_channels=temporal_ts_channels,
    temporal_ts_kernel_sizes=temporal_ts_kernel_sizes,
    spatial_agg_channels=spatial_agg_channels,
    spatial_agg_kernel_size_z=spatial_agg_kernel_size_z,
    spatial_agg_kernel_size_xy=spatial_agg_kernel_size_xy,
    spatial_agg_stride_z=spatial_agg_stride_z,
    spatial_agg_stride_xy=spatial_agg_stride_xy,
    spatial_agg_pool_kernel_z=spatial_agg_pool_kernel_z,
    spatial_agg_pool_kernel_xy=spatial_agg_pool_kernel_xy,
    spatial_agg_pool_stride_z=spatial_agg_pool_stride_z,
    spatial_agg_pool_stride_xy=spatial_agg_pool_stride_xy,
    patch_size_z=patch_size_z,
    patch_size_xy=patch_size_xy,
    embedding_dim=embedding_dim,
    num_prototypes=num_prototypes,
    consistency_weight=consistency_weight,
    feature_weight=feature_weight,
    prototype_temperature=prototype_temperature,
    dropout=dropout,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    validation_split=0.1,
    random_state=random_seed,
    device=device,
    verbose=True,
)
model.fit(X_train, y_train, validation_data=(X_val, y_val))


In [ ]:
plot_training_history(model, title="Commutative CNN train/validation loss");


In [ ]:
holdout_pred = model.predict(X_holdout)
report_df = pd.DataFrame(
    classification_report(
        y_holdout,
        holdout_pred,
        labels=model.classes_,
        target_names=[dataset["label_map"][int(label)] for label in model.classes_],
        output_dict=True,
        zero_division=0,
    )
).T
display(report_df)
plot_confusion_matrices(
    y_holdout,
    holdout_pred,
    class_labels=model.classes_,
    label_map=dataset["label_map"],
);

In [ ]:
val_loss_components = pd.DataFrame([model.evaluate_loss_components(X_val, y_val)])
holdout_loss_components = pd.DataFrame([model.evaluate_loss_components(X_holdout, y_holdout)])
val_loss_components.insert(0, "split", "val")
holdout_loss_components.insert(0, "split", "holdout")
display(pd.concat([val_loss_components, holdout_loss_components], ignore_index=True))


In [ ]:
holdout_branch_embeddings = model.transform_branches(X_holdout)
branch_alignment_df = metadata_holdout.copy()
branch_alignment_df["st_ts_l2_distance"] = np.linalg.norm(
    holdout_branch_embeddings["st_embedding"] - holdout_branch_embeddings["ts_embedding"],
    axis=1,
)
display(branch_alignment_df[["label_name", "compound", "st_ts_l2_distance"]].head(10))

holdout_embeddings = torch.from_numpy(holdout_branch_embeddings["embedding"])
holdout_embedding_df = build_tensor_embedding_2d(
    holdout_embeddings,
    y_holdout,
    label_map=dataset["label_map"],
    metadata=metadata_holdout,
    method=embedding_method,
    random_state=random_seed,
    umap_n_neighbors=umap_n_neighbors,
    umap_min_dist=umap_min_dist,
)
display(holdout_embedding_df.head(5))
plot_tensor_embedding_2d(
    holdout_embedding_df,
    title=f"Holdout learned embeddings | {embedding_method.upper()}",
    marker_column="compound",
    show_svm_background=True,
);
